# Task 1 - Anti-Cheat Pre-Filter (binary, metric: F2 macro)

5-fold LightGBM with account-credibility and aim-vs-game-sense features; the decision threshold maximises out-of-fold macro F2. Run from inside `task1/`.

In [1]:
import pandas as pd
import numpy as np

# run from inside task1/
train_t1 = pd.read_csv('train.csv')
test_t1 = pd.read_csv('test.csv')

print("=== shapes ===")
print(f"Train: {train_t1.shape} | Test: {test_t1.shape}")

print("\n=== is_cheater counts (incl. missing) ===")
print(train_t1['is_cheater'].value_counts(dropna=False))

print("\n=== feature means: legitimate (0) vs cheater (1) ===")
summary = train_t1.groupby('is_cheater').mean(numeric_only=True).T
summary['diff_ratio'] = summary[1.0] / (summary[0.0] + 1e-5)
print(summary.sort_values(by='diff_ratio', ascending=False).round(3))

=== shapes ===
Train: (104000, 33) | Test: (25889, 32)

=== is_cheater counts (incl. missing) ===
is_cheater
0.0    59091
1.0    31725
NaN    13184
Name: count, dtype: int64

=== feature means: legitimate (0) vs cheater (1) ===
is_cheater                   0.0      1.0  diff_ratio
reaction_time_ms         133.953  137.670       1.028
device_changes_count       7.438    7.474       1.005
night_play_ratio           0.378    0.380       1.005
team_play_score            0.644    0.646       1.003
clutch_success_rate        0.340    0.340       1.002
movement_pattern_score     0.714    0.715       1.001
weapon_switch_speed        0.634    0.634       1.000
communication_rate         0.550    0.550       1.000
sessions_per_day           4.945    4.947       1.000
utility_usage_rate         0.581    0.581       1.000
survival_time_avg         66.239   66.183       0.999
map_knowledge              0.690    0.689       0.999
buy_decision_score         0.598    0.597       0.999
first_blood_rate

In [2]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score

# labeled rows only
train_labeled_t1 = train_t1.dropna(subset=['is_cheater']).reset_index(drop=True)
features_t1 = [col for col in test_t1.columns if col != 'player_id']

X_t1 = train_labeled_t1[features_t1]
y_t1 = train_labeled_t1['is_cheater'].astype(int)

# baseline: 5-fold LightGBM
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_t1 = np.zeros(len(train_labeled_t1))
importances = np.zeros(len(features_t1))

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'n_estimators': 1000,
    'random_state': 42,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_t1, y_t1)):
    X_tr, y_tr = X_t1.iloc[train_idx], y_t1.iloc[train_idx]
    X_va, y_va = X_t1.iloc[val_idx], y_t1.iloc[val_idx]

    model_t1 = lgb.LGBMClassifier(**lgb_params)
    model_t1.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_preds_t1[val_idx] = model_t1.predict_proba(X_va)[:, 1]
    importances += model_t1.feature_importances_ / skf.n_splits

# threshold that maximises OOF macro F2
best_thresh_t1 = 0.5
best_f2_t1 = 0.0
for thresh in np.arange(0.10, 0.90, 0.01):
    score = fbeta_score(y_t1, (oof_preds_t1 >= thresh).astype(int), beta=2, average='macro')
    if score > best_f2_t1:
        best_f2_t1 = score
        best_thresh_t1 = thresh

print("=== baseline ===")
print(f"Best Threshold : {best_thresh_t1:.2f}")
print(f"Macro F2 Score : {best_f2_t1:.5f}")

print("\n=== top 15 features ===")
imp_df = pd.DataFrame({'feature': features_t1, 'importance': importances})
print(imp_df.sort_values(by='importance', ascending=False).head(15).to_string(index=False))

C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


=== baseline ===
Best Threshold : 0.37
Macro F2 Score : 0.77872

=== top 15 features ===
                feature  importance
       account_age_days      1039.0
    friend_network_size      1022.4
       reports_received       653.0
    crosshair_placement       539.6
                  level       491.2
       game_sense_score       449.8
    headshot_percentage       389.0
       kill_death_ratio       379.8
level_progression_speed       376.4
               win_rate       317.4
         accuracy_score       286.0
       kill_consistency       285.0
    spray_control_score       274.4
     buy_decision_score       265.4
      aiming_smoothness       262.4


In [3]:
def add_features(df):
    df = df.copy()

    # account credibility
    df['friends_per_day'] = df['friend_network_size'] / (df['account_age_days'] + 1.0)
    df['reports_per_day'] = df['reports_received'] / (df['account_age_days'] + 1.0)
    df['reports_per_level'] = df['reports_received'] / (df['level'] + 1.0)
    df['level_per_day'] = df['level'] / (df['account_age_days'] + 1.0)

    # aim vs game sense imbalance
    df['aim_to_gamesense_ratio'] = (df['headshot_percentage'] + df['accuracy_score']) / (df['game_sense_score'] + df['map_knowledge'] + 1e-5)
    df['kd_to_winrate_ratio'] = df['kill_death_ratio'] / (df['win_rate'] + 1.0)

    # missing values per player
    base_cols = [c for c in test_t1.columns if c != 'player_id']
    df['missing_count'] = df[base_cols].isnull().sum(axis=1)

    return df

# feature-engineered training set
train_fe_t1 = add_features(train_labeled_t1)
features_fe_t1 = [col for col in train_fe_t1.columns if col not in ['player_id', 'is_cheater']]

X_fe_t1 = train_fe_t1[features_fe_t1]
y_fe_t1 = train_fe_t1['is_cheater'].astype(int)

# same 5-fold setup and parameters as the baseline
oof_fe_t1 = np.zeros(len(train_fe_t1))
imp_fe = np.zeros(len(features_fe_t1))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fe_t1, y_fe_t1)):
    X_tr, y_tr = X_fe_t1.iloc[train_idx], y_fe_t1.iloc[train_idx]
    X_va, y_va = X_fe_t1.iloc[val_idx], y_fe_t1.iloc[val_idx]

    model_fe = lgb.LGBMClassifier(**lgb_params)
    model_fe.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_fe_t1[val_idx] = model_fe.predict_proba(X_va)[:, 1]
    imp_fe += model_fe.feature_importances_ / skf.n_splits

best_thresh_fe = 0.5
best_f2_fe = 0.0
for thresh in np.arange(0.10, 0.90, 0.01):
    score = fbeta_score(y_fe_t1, (oof_fe_t1 >= thresh).astype(int), beta=2, average='macro')
    if score > best_f2_fe:
        best_f2_fe = score
        best_thresh_fe = thresh

print("=== feature engineering vs baseline ===")
print(f"baseline macro F2 : {best_f2_t1:.5f}")
print(f"FE macro F2       : {best_f2_fe:.5f} ({best_f2_fe - best_f2_t1:+.5f})")
print(f"best threshold    : {best_thresh_fe:.2f}")

print("\n=== top 15 features (FE) ===")
imp_fe_df = pd.DataFrame({'feature': features_fe_t1, 'importance': imp_fe})
print(imp_fe_df.sort_values(by='importance', ascending=False).head(15).to_string(index=False))

C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


=== feature engineering vs baseline ===
baseline macro F2 : 0.77872
FE macro F2       : 0.77885 (+0.00013)
best threshold    : 0.40

=== top 15 features (FE) ===
                feature  importance
    friend_network_size       968.0
       account_age_days       901.6
       reports_received       559.2
    crosshair_placement       551.8
                  level       445.2
       game_sense_score       439.6
    headshot_percentage       356.4
level_progression_speed       341.0
       kill_death_ratio       317.2
               win_rate       306.6
       kill_consistency       282.6
    spray_control_score       282.0
      aiming_smoothness       269.2
         accuracy_score       262.4
       reaction_time_ms       258.8


In [4]:
from sklearn.metrics import fbeta_score

# shared submission file at the repo root
SUB_PATH = '../submission_final.csv'
sub_before = pd.read_csv(SUB_PATH)

# test features
test_fe_t1 = add_features(test_t1)
X_test_fe_t1 = test_fe_t1[features_fe_t1]

assert len(X_test_fe_t1) == len(sub_before), "test rows do not match the submission"

# 5-fold training, test prediction = mean of the fold models
oof_final = np.zeros(len(X_fe_t1))
test_preds_final = np.zeros(len(X_test_fe_t1))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_fe_t1, y_fe_t1)):
    X_tr, y_tr = X_fe_t1.iloc[train_idx], y_fe_t1.iloc[train_idx]
    X_va, y_va = X_fe_t1.iloc[val_idx], y_fe_t1.iloc[val_idx]

    model_final = lgb.LGBMClassifier(**lgb_params)
    model_final.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_final[val_idx] = model_final.predict_proba(X_va)[:, 1]
    test_preds_final += model_final.predict_proba(X_test_fe_t1)[:, 1] / skf.n_splits

# threshold that maximises OOF macro F2
best_thresh_final = 0.5
best_f2_final = 0.0
for thresh in np.arange(0.10, 0.90, 0.01):
    score = fbeta_score(y_fe_t1, (oof_final >= thresh).astype(int), beta=2, average='macro')
    if score > best_f2_final:
        best_f2_final = score
        best_thresh_final = thresh

print(f"Best Threshold (Macro F2): {best_thresh_final:.2f} | Score: {best_f2_final:.5f}")

# write the task1 column only; other columns must stay unchanged
sub_after = sub_before.copy()
sub_after['task1'] = (test_preds_final >= best_thresh_final).astype(int)

other_cols = ['id', 'task2', 'task3', 'task4', 'task5']
is_safe = sub_before[other_cols].equals(sub_after[other_cols])

print("other columns unchanged:", is_safe)
print("task1 prediction shares:\n", sub_after['task1'].value_counts(normalize=True).round(4))

if is_safe and sub_after['task1'].isnull().sum() == 0:
    sub_after.to_csv(SUB_PATH, index=False)
    print(f"saved task1 to {SUB_PATH}")

C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


C:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best Threshold (Macro F2): 0.40 | Score: 0.77885
other columns unchanged: True
task1 prediction shares:
 task1
0    0.6171
1    0.3829
Name: proportion, dtype: float64
saved task1 to ../submission_final.csv
